<a href="https://colab.research.google.com/github/santiagodie/sudoku-vision-solver/blob/main/02_digit_recognition_and_solving.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Digit recognition and solving

This notebook completes the pipeline started in [notebook 01](01_grid_detection.ipynb): it reads the digit in each filled cell, solves the puzzle and draws the solution back onto the original image.

## Setup

In [ ]:
import glob, os, time
import cv2
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
%matplotlib inline

BOARDS = ["data/boards/sudoku1.png", "data/boards/sudoku2.png", "data/boards/sudoku3.png"]

## 1. Grid detection

The same steps as notebook 01, condensed: binarise, keep the 81 square contours and sort them row by row. `show_image` and `sort_contours` come from the course material.

In [ ]:
# Provided in the course material.
def show_image(image):
    if len(image.shape) == 3:
        plt.imshow(image[:, :, ::-1])
    else:
        plt.imshow(image, cmap="gray")
    plt.axis("off")
    plt.show()

# Provided in the course material.
def sort_contours(cnts):
    boundingBoxes = [cv2.boundingRect(c) for c in cnts]
    (cnts, boundingBoxes) = zip(*sorted(zip(cnts, boundingBoxes), key=lambda b: (b[1][1], b[1][0]), reverse=False))
    return cnts

def is_cell(c):
    area = cv2.contourArea(c)
    x, y, w, h = cv2.boundingRect(c)
    solidity = area / float(cv2.contourArea(cv2.convexHull(c)))
    return solidity > 0.9 and 0.8 < w / float(h) < 1.2 and 1000 < area < 15000

def detect_grid(image_path):
    """Return the image, its binarised version (white ink on black) and the 81 sorted cells."""
    image = cv2.imread(image_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    thresh = cv2.threshold(gray, 30, 255, cv2.THRESH_BINARY_INV)[1]
    contours, _ = cv2.findContours(thresh, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    cells = [c for c in contours if is_cell(c)]
    return image, thresh, sort_contours(cells)

def crop_cell(image, c, margin=5):
    x, y, w, h = cv2.boundingRect(c)
    return image[y + margin:y + h - margin, x + margin:x + w - margin]

def is_empty(cell_crop):
    # A few stray pixels are tolerated; a digit has far more than 20.
    return np.count_nonzero(cell_crop) < 20

def print_grid(board_string):
    for r in range(9):
        row = board_string[r * 9:(r + 1) * 9]
        print(" ".join(row[0:3]), "|", " ".join(row[3:6]), "|", " ".join(row[6:9]))
        if r in (2, 5):
            print("------+-------+------")

## 2. First attempt: a Random Forest trained on handwritten digits

An earlier course exercise trained a Random Forest on a dataset of **2,560 handwritten digits** (256 per class) in MNIST format: 28×28 grayscale images, white digit on black. The original model was saved as a 28 MB pickle; here it is retrained with the same hyperparameters, which takes a few seconds and avoids loading a pickled model.

Each image is described by its raw pixels, flattened into a 784-value vector (the `RawImage` descriptor from the course material).

In [ ]:
# Loading adapted from the course material (CargaYDescribe + RawImage descriptor).
def load_digits(folder="data/digits"):
    data, labels = [], []
    for path in sorted(glob.glob(f"{folder}/*/*.jpg")):
        data.append(cv2.imread(path, cv2.IMREAD_GRAYSCALE).flatten())
        labels.append(os.path.basename(os.path.dirname(path)))
    return np.array(data), np.array(labels)

data, labels = load_digits()
print("Dataset:", data.shape)

samples = [cv2.imread(sorted(glob.glob(f"data/digits/{d}/*.jpg"))[0], cv2.IMREAD_GRAYSCALE) for d in range(10)]
show_image(np.hstack(samples))

In [ ]:
trainData, testData, trainLabels, testLabels = train_test_split(
    data, labels, test_size=0.25, random_state=84, stratify=labels)

# Hyperparameters of the model from the earlier exercise.
start = time.time()
model = RandomForestClassifier(n_estimators=406, max_depth=30, min_samples_split=4,
                               max_features="sqrt", random_state=42, n_jobs=-1)
model.fit(trainData, trainLabels)
print(f"Trained in {time.time() - start:.1f} s\n")
print(classification_report(testLabels, model.predict(testData), digits=3))

Around 92% accuracy on handwritten digits. The first version applied it directly to the board, resizing each whole cell to 28×28:

In [ ]:
def read_board_rf(image_path, prepare):
    _, thresh, cells = detect_grid(image_path)
    board = ""
    for c in cells:
        crop = crop_cell(thresh, c)
        board += "." if is_empty(crop) else model.predict(prepare(crop).reshape(1, -1))[0]
    return board

def whole_cell(crop):
    return cv2.resize(crop, (28, 28), interpolation=cv2.INTER_AREA)

print_grid(read_board_rf(BOARDS[0], whole_cell))

Almost every digit is read as 1 or 9. The model was trained on handwriting and these boards use a printed typeface, so the next section switches to an OCR engine built for printed text.